# 2. (선택) 코드로 학습하기 — GPU 필요
강의장에서는 **Unsloth Studio**(화면 클릭)로 학습해요. 이 노트북은 **같은 설정을 코드로** 돌리는 방법이에요. 결과는 같아요.

| 어디서 | 준비 |
|---|---|
| Colab | 런타임 → 런타임 유형 변경 → **T4 GPU** |
| 강의장 PC (VS Code) | `.venv`에 **GPU판 PyTorch**(`+cu128`)와 unsloth — 아래 「0. 확인」 |

준비물: `train.jsonl` · `eval.jsonl` · `test.jsonl` (1번 노트북에서 만든 파일)  
만들어지는 것: `outputs/`(중간 저장) · `lead_model_lora/`(학습한 부품) · `lead_model_gguf/`(Ollama용)  
⚠ 학습한 모델은 GitHub · Hugging Face 등 온라인에 올리지 않아요.

In [ ]:
# ▶ 환경 준비 — Colab · VS Code 공통, 맨 처음 한 번 실행
import os, sys, platform, subprocess
IN_COLAB = 'google.colab' in sys.modules

def gpu_info():
    try:
        r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                           capture_output=True, text=True, timeout=10)
        return r.stdout.strip() or None
    except Exception:
        return None

GPU = gpu_info()
print('실행 위치:', 'Colab' if IN_COLAB else f'내 컴퓨터 ({platform.system()})')
print('GPU:', GPU or '없음 → 이 노트북은 GPU가 필요해요')

# (선택) pandas를 GPU로 — Colab GPU 런타임 또는 Linux/WSL에 cuDF가 있을 때만
USE_GPU_PANDAS = False          # True로 바꾸고 이 칸을 다시 실행
if USE_GPU_PANDAS and GPU:
    try:
        get_ipython().run_line_magic('load_ext', 'cudf.pandas')
        print('pandas → GPU(cuDF) 사용')
    except Exception as e:
        print('cuDF를 쓸 수 없어 일반 pandas로 실행:', type(e).__name__)

# 한글 글꼴: Colab은 나눔고딕 설치, 윈도우는 맑은 고딕, 맥은 애플고딕
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
if IN_COLAB:
    nanum = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if not os.path.exists(nanum):
        subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null', shell=True)
    fm.fontManager.addfont(nanum)
    plt.rcParams['font.family'] = 'NanumGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:                                   # 리눅스·WSL: 설치된 한글 글꼴 찾기
    ko = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name or 'CJK KR' in f.name]
    if ko: plt.rcParams['font.family'] = ko[0]
plt.rcParams['axes.unicode_minus'] = False
print('한글 글꼴:', plt.rcParams['font.family'])

## 0. 확인 — GPU판 PyTorch와 unsloth
⚠ **`pip install unsloth`를 그냥 실행하지 마세요.** Windows에서는 PyTorch가 **CPU판(`+cpu`)으로 바뀌어** GPU를 못 쓰게 될 수 있어요.
아래 칸이 `+cu128 True`와 unsloth 버전을 보여 주면 바로 「1.」로 가세요.

In [ ]:
assert GPU, 'GPU가 없어요. Colab은 T4 GPU 런타임으로 바꾸고, 강의장 PC는 Unsloth Studio를 쓰세요.'
import torch
print('PyTorch', torch.__version__, '| CUDA 사용 가능:', torch.cuda.is_available())
try:
    import unsloth
    print('unsloth', unsloth.__version__, '✅')
except Exception as e:
    print('unsloth 없음 또는 불러오기 실패 →', type(e).__name__, '| 아래 칸 안내를 따르세요')

In [ ]:
# ▶ 위에서 unsloth가 없을 때만 — 주석(#)을 지우고 '두 줄 모두' 실행 → Restart
# Colab은 첫 줄만 실행하면 돼요.
# %pip install unsloth
# %pip install --force-reinstall --no-deps torch torchvision --index-url https://download.pytorch.org/whl/cu128   # PyTorch를 GPU판으로 되돌리기

## 1. 기본 모델 불러오기 — 4bit로 작게 (QLoRA)
큰 모델은 얼려 두고, 작은 부품(LoRA, 전체의 약 1%)만 학습해요.

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-3B-Instruct-bnb-4bit',
    max_seq_length=2048,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth', random_state=42,
)

## 2. 데이터를 대화 형식으로 바꾸기
`instruction + input`은 사용자 말, `output`은 모델의 답이 돼요.

In [ ]:
from datasets import load_dataset
ds = load_dataset('json', data_files={'train': 'train.jsonl', 'eval': 'eval.jsonl'})

def to_text(ex):
    msgs = [{'role': 'user', 'content': ex['instruction'] + '\n\n' + ex['input']},
            {'role': 'assistant', 'content': ex['output']}]
    return {'text': tokenizer.apply_chat_template(msgs, tokenize=False)}

ds = ds.map(to_text)
print(ds)

## 3. 학습 — Studio와 같은 설정
에포크 2 · 배치 2 × 누적 4 · 학습률 2e-4 · rank 16 / alpha 32. 강의장 PC에서 약 5분 걸려요.
진행 막대의 스텝 수 = ⌈train 개수 ÷ 8⌉ × 2

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=ds['train'], eval_dataset=ds['eval'],
    args=SFTConfig(
        dataset_text_field='text', max_seq_length=2048,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=2, learning_rate=2e-4,
        optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='linear',
        logging_steps=1, eval_strategy='epoch',
        seed=42, output_dir='outputs', report_to='none',
    ),
)
stats = trainer.train()

In [ ]:
# loss 그래프 — train은 내려가는데 eval이 올라가면 외우기 시작(과적합)
import pandas as pd
log = pd.DataFrame(trainer.state.log_history)
ax = log.dropna(subset=['loss']).plot(x='step', y='loss', label='train', figsize=(8, 4))
if 'eval_loss' in log:
    log.dropna(subset=['eval_loss']).plot(x='step', y='eval_loss', label='eval', ax=ax, marker='o')
ax.set_ylabel('loss'); plt.show()
print(log[['epoch', 'loss', 'eval_loss']].dropna(how='all', subset=['loss', 'eval_loss']).tail(3))

## 4. 시험 문제로 확인 — 본문에 없는 숫자·원인을 지어내지 않았나?

In [ ]:
FastLanguageModel.for_inference(model)
test = pd.read_json('test.jsonl', lines=True)
for i, r in test.iterrows():
    msgs = [{'role': 'user', 'content': r['instruction'] + '\n\n' + r['input']}]
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                           return_tensors='pt', return_dict=True).to('cuda')
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    answer = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'[{i}] 모델: {answer}\n    정답: {r["output"]}\n' + '-' * 60)

## 5. 저장 · GGUF로 내보내기
- `lead_model_lora/` : 학습한 부품만 (수십 MB) — 나중에 이어서 학습할 때
- `lead_model_gguf/` : Ollama용 GGUF 파일 + `Modelfile` (약 2GB, 몇 분 걸려요)

In [ ]:
model.save_pretrained('lead_model_lora'); tokenizer.save_pretrained('lead_model_lora')
model.save_pretrained_gguf('lead_model', tokenizer, quantization_method='q4_k_m')
import glob
print('만들어진 파일:', glob.glob('lead_model_gguf/*'))

## 6. Ollama에 등록하기 (터미널)
```powershell
cd lead_model_gguf
ollama create mbcsports -f Modelfile
ollama list
```
`Modelfile` 첫 줄이 `FROM ./Qwen2.5-3B-Instruct.Q4_K_M.gguf`처럼 GGUF 파일을 가리키는지 먼저 확인하세요.  
`ollama create`는 **내 PC 안에만** 등록해요. 온라인 업로드(`ollama push`)는 이 수업에서 쓰지 않아요.  
다음: `4_compare_results.ipynb`의 `MY_MODEL`을 등록한 이름으로 바꾸고 비교해요.